In [22]:
import numpy as np
import xtrack as xt
import matplotlib.pyplot as plt
from scipy.constants import c as clight

## 1. Load the line and enable radiation

In [23]:
env = xt.load('fccee_z.json')
line = env['fccee_p_ring']
print('p0c [GeV]:', line.particle_ref.p0c[0]/1e9, ' length [m]:', line.get_length())
cav = line['rf400']
print('RF: V=%.3g V, f=%.6g Hz, lag=%g deg' % (cav.voltage, cav.frequency, cav.lag))

line.configure_radiation(model='mean')

Loading line from dict: 100%|██████████| 19024/19024 [00:05<00:00, 3278.29it/s]


p0c [GeV]: 45.59999999713684  length [m]: 90660.25633484735
RF: V=1.5e+08 V, f=4.0078e+08 Hz, lag=0 deg


# 2. Compute SR damping times

In [31]:
tw = line.twiss(radiation_integrals=True, eneloss_and_damping=True)

alpha_turn = tw.damping_constants_turns   
alpha_s    = tw.damping_constants_s       
t_rev      = tw.T_rev0

tau_turns = 1/alpha_turn                  
print('T_rev [us]            :', t_rev*1e6)
print('damping const [1/turn]:', alpha_turn)
print('damping const [1/s]   :', alpha_s)
print('amplitude damping time [turns] x,y,z:', tau_turns)
print('amplitude damping time [ms]    x,y,z:', 1e3/alpha_s)
print('emittance damping time [turns] x,y,z:', tau_turns/2)
print('partition numbers Jx,Jy,Jz:', tw.partition_numbers, ' sum =', tw.partition_numbers.sum())
print('energy loss / turn [GeV]:', tw.eneloss_turn/1e6)
print('equilibrium gemitt x,y,zeta:', tw.eq_gemitt_x, tw.eq_gemitt_y, tw.eq_gemitt_zeta)

/Users/skostogl/Documents/workspace/workspace_2025/python_installation/miniforge3/envs/FCC_xsuite/lib/python3.12/site-packages/xtrack/twiss/twiss_defaults_and_input_preparation.py:501: FutureWarning: The `eneloss_and_damping` keyword is deprecated and will be removed in future versions. 
Please use `radiation_analysis` instead, which has the same behavior. This deprecation is part of the interface cleanup in view of the 1.0 release.
  warn('The `eneloss_and_damping` keyword is deprecated and will be removed in future versions. \n'


T_rev [us]            : 302.4100637666468
damping const [1/turn]: [0.00038196 0.00038206 0.00076425]
damping const [1/s]   : [1.26306725 1.26339745 2.52719553]
amplitude damping time [turns] x,y,z: [2618.0460866  2617.36183644 1308.47345233]
amplitude damping time [ms]    x,y,z: [791.72348399 791.51655986 395.69554016]
emittance damping time [turns] x,y,z: [1309.0230433  1308.68091822  654.23672616]
partition numbers Jx,Jy,Jz: [0.9997255  0.99998685 2.00029082]  sum = 4.000003167618834
energy loss / turn [GeV]: 34.84470851783254
equilibrium gemitt x,y,zeta: 1.2169638414057277e-09 4.676194134050793e-15 1.6451557922417808e-06


/Users/skostogl/Documents/workspace/workspace_2025/python_installation/miniforge3/envs/FCC_xsuite/lib/python3.12/site-packages/xtrack/table.py:317: FutureWarning: `T_rev0` is deprecated, use `t_rev0` instead. This deprecation is part of the interface cleanup in view of the 1.0 release.
  warn(depr_fields[name], FutureWarning)
/Users/skostogl/Documents/workspace/workspace_2025/python_installation/miniforge3/envs/FCC_xsuite/lib/python3.12/site-packages/xtrack/table.py:317: FutureWarning: `eneloss_turn` is deprecated, use `energy_loss` instead. This deprecation is part of the interface cleanup in view of the 1.0 release.
  warn(depr_fields[name], FutureWarning)


In [32]:
E0 = line.particle_ref.energy0[0]
U0 = tw.eneloss_turn
J  = tw.partition_numbers
print('from radiation integrals [1/s]:', tw.rad_int_damping_constant_x_s, tw.rad_int_damping_constant_y_s, tw.rad_int_damping_constant_zeta_s)
print('one-turn matrix        [1/s]:', alpha_s)
print('textbook tau [turns] 2E0/(J U0):', 2*E0/(J*U0))
print('xsuite   tau [turns]           :', tau_turns)

from radiation integrals [1/s]: 1.2632103866371402 1.2634144244087357 2.527032880805214
one-turn matrix        [1/s]: [1.26306725 1.26339745 2.52719553]
textbook tau [turns] 2E0/(J U0): [2618.04576085 2617.36151078 1308.47328952]
xsuite   tau [turns]           : [2618.0460866  2617.36183644 1308.47345233]


# 3. Compute IBS amplitude growth times

In [37]:
# parameters from here: https://gitlab.cern.ch/acc-models/fcc/acc-models-fcc-ee/-/blob/dev/108.0.0/parameter_tables/z_parameter_table.txt?ref_type=heads

Nb      = 2.175979e11       # particles per bunch
gemitt_x = 1.21494e-9 #tw.eq_gemitt_x   # use the SR equilibrium horizontal emittance
gemitt_y = 2.419171e-12 #1.9e-12          # [m] assumed vertical emittance (SR-only eq. is ~ 0, set by coupling)
gemitt_z = tw.eq_gemitt_zeta
betz0    = tw.bets0  # longitudinal beta (xsuite uses bets0 in the IBS evolution)
sigma_z     =  15.082817*1e-3 #np.sqrt(gemitt_z*betz0)
sigma_delta = 0.127394*1e-2#np.sqrt(gemitt_z/betz0)
print('sigma_z [mm] = %.3f, sigma_delta = %.3e' % (sigma_z*1e3, sigma_delta))

ibs = tw.get_ibs_growth_rates(
    formalism='Nagaitsev',
    total_beam_intensity=Nb,
    gemitt_x=gemitt_x, gemitt_y=gemitt_y,
    sigma_delta=sigma_delta, bunch_length=sigma_z,
    bunched=True)

K = np.array([ibs.Kx, ibs.Ky, ibs.Kz])      # amplitude rates [1/s]
print('IBS amplitude growth rates [1/s]:', K)
print('IBS amplitude growth time  [s]  :', 1/K)
print('IBS amplitude growth time  [turns]:', 1/(K*t_rev))
print('IBS emittance growth time  [turns]:', 1/(2*K*t_rev))

Vertical dispersion is present, Nagaitsev formalism does not account for it


sigma_z [mm] = 15.083, sigma_delta = 1.274e-03
IBS amplitude growth rates [1/s]: [1.30402353e-02 1.76404595e-05 2.88432902e-03]
IBS amplitude growth time  [s]  : [   76.68573276 56687.86590978   346.70108533]
IBS amplitude growth time  [turns]: [2.53581947e+05 1.87453636e+08 1.14646014e+06]
IBS emittance growth time  [turns]: [  126790.97349412 93726817.82429004   573230.07212161]


# Ratio

In [38]:
tau_turns[0] / (1/(K*t_rev)[0]) *100.

np.float64(1.0324260530749652)